# Análisis de Víctimas Fatales en Siniestros Viales · CABA 2019–2024
### Proyecto de Portafolio — Ciencia de Datos Aplicada a Política Pública

---
**Autor:** David Palacio Velásquez  
**Dataset:** Víctimas en siniestros viales — Buenos Aires Data (GCBA)  
**Fuente oficial:** [data.buenosaires.gob.ar](https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales)  
**Período:** 2019–2024 · 610 víctimas fatales · Datos oficiales del Observatorio de Movilidad y Seguridad Vial (OMSV)

---

## 1. Introducción y Objetivo del Análisis

Los siniestros viales son la **primera causa de muerte violenta en Argentina**. En la Ciudad de Buenos Aires, entre 2019 y 2024 fallecieron **610 personas** en accidentes de tránsito — más de 100 muertes por año en promedio.

Estas muertes no son eventos aleatorios ni inevitables: responden a **patrones sistemáticos** identificables con datos. El dataset utilizado proviene del **Observatorio de Movilidad y Seguridad Vial (OMSV)** del GCBA, publicado en el portal de datos abiertos Buenos Aires Data bajo licencia Creative Commons. Contiene todos los siniestros viales con víctimas (leves, graves y fatales) registrados por la Policía de la Ciudad.

### Preguntas que guían este análisis

1. **¿Quiénes mueren?** — Perfil de la víctima: modo de desplazamiento, sexo, edad
2. **¿Cómo evolucionó?** — Tendencias 2019–2024, incluyendo el efecto COVID-19
3. **¿Hay diferencias por perfil etario?** — ¿A qué edad muere cada tipo de víctima?
4. **¿Qué rol cumple el género?** — Distribución por sexo dentro de cada grupo
5. **¿Qué implicaciones tiene para la política pública?**

### Nota metodológica sobre el dataset

El archivo original contiene **62.076 registros** (leves, graves y fatales). Este análisis se enfoca exclusivamente en las **610 víctimas con gravedad MORTAL**, que representan el universo de fallecidos en siniestros viales del período. Esta decisión está justificada por el objetivo del análisis: identificar patrones en muertes viales, no en lesiones.

## 2. Carga y Comprensión del Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style('whitegrid')

PALETA     = ['#2166ac','#d6604d','#4dac26','#f4a582','#762a83','#1b7837']
COLOR_PRIM = '#2166ac'
COLOR_SEC  = '#d6604d'

In [ ]:
# ── Carga del dataset completo ─────────────────────────────────────────────────
df_raw = pd.read_csv('siniestros_viales_victimas.csv', sep=None, engine='python')

print(f"Dataset completo: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Columnas: {df_raw.columns.tolist()}")
df_raw.head(3)

### ¿Qué contiene el dataset?

| Variable | Tipo | Descripción |
|---|---|---|
| `id_siniestro` | str | Identificador único del siniestro (`HC-YYYY-XXXXXXX`) |
| `fecha_siniestro` | str → datetime | Fecha del hecho |
| `anio_siniestro` | float | Año del siniestro |
| `modo_desplazamiento_victima` | str | Modo de transporte de la víctima |
| `sexo_victima` | str | Sexo (`M` / `F` / `SD`) |
| `edad_victima` | str | Edad de la víctima |
| `GRAVEdad_victima` | str | Gravedad: `LEVE` / `GRAVE` / `MORTAL` |
| `rol_victima` | str | Rol en el siniestro |
| `fecha_fallecimiento_victima` | str | Fecha de fallecimiento (solo para MORTAL) |

**Nota:** el dataset incluye víctimas de todos los niveles de gravedad. Este análisis filtra exclusivamente las **610 víctimas MORTAL**.

## 3. Limpieza y Preparación de Datos

In [ ]:
# ── Paso 1: eliminar filas vacías ──────────────────────────────────────────────
# El CSV tiene ~986.000 filas completamente vacías (artifact del formato Excel original)
df_clean = df_raw.dropna(subset=['id_siniestro']).copy()
print(f"Filas con datos reales: {len(df_clean):,} (de {len(df_raw):,} totales)")

# ── Paso 2: filtrar solo víctimas MORTALES ──────────────────────────────────────
df = df_clean[df_clean['GRAVEdad_victima'] == 'MORTAL'].copy()
print(f"Víctimas fatales: {len(df):,}")
print(f"Distribución por gravedad:")
print(df_clean['GRAVEdad_victima'].value_counts())

In [ ]:
# ── Paso 3: normalización y variables derivadas ─────────────────────────────────
df['fecha_dt'] = pd.to_datetime(df['fecha_siniestro'], errors='coerce')
df['anio']     = df['anio_siniestro'].astype(int)
df['mes']      = df['fecha_dt'].dt.month

# Normalizar texto (mayúsculas y strip) para unificar variantes
df['modo'] = df['modo_desplazamiento_victima'].str.upper().str.strip()
df['sexo'] = df['sexo_victima'].str.upper().str.strip()
df['rol']  = df['rol_victima'].str.upper().str.strip()

# Edad numérica (viene como string en el original)
df['edad'] = pd.to_numeric(df['edad_victima'], errors='coerce')

# Edad imputada con mediana para usos en modelo
mediana_edad = df['edad'].median()
df['edad_imp'] = df['edad'].fillna(mediana_edad)

# Verificar
print(f"Período: {df['anio'].min()} – {df['anio'].max()}")
print(f"Missing en edad: {df['edad'].isna().sum()} ({df['edad'].isna().mean():.1%})")
print(f"Mediana de edad (imputación): {mediana_edad:.0f} años")
df[['id_siniestro','fecha_dt','anio','modo','sexo','edad','rol']].head()

**Decisiones de limpieza justificadas:**

- **Filas vacías (~986K):** eliminadas — son el resultado de exportar un Excel con celdas en blanco a CSV. No representan datos.
- **Filtro MORTAL:** el análisis se enfoca en fallecidos. Incluir leves y graves mezclaría poblaciones con dinámicas distintas y diluiría los patrones de riesgo fatal.
- **Normalización de texto:** el campo `rol_victima` tiene variantes de capitalización (`Conductor`, `conductor`, `CONDUCTOR`). Unificadas a mayúsculas.
- **Edad como string:** el campo viene como texto en el original (posiblemente por valores como `SD`). Convertido a numérico con `errors='coerce'`.

## 4. Análisis Exploratorio (EDA)

### 4.1 · Evolución anual de víctimas fatales

In [ ]:
por_anio = df.groupby('anio').size().reset_index(name='n')
media    = por_anio['n'].mean()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.fill_between(por_anio['anio'], por_anio['n'], alpha=0.12, color=COLOR_PRIM)
ax.plot(por_anio['anio'], por_anio['n'],
        marker='o', color=COLOR_PRIM, linewidth=2.5, markersize=9, zorder=5)

for _, row in por_anio.iterrows():
    offset = 12 if row['n'] > media else -18
    color  = COLOR_SEC if row['n'] == por_anio['n'].min() else '#333'
    ax.annotate(f"{int(row['n'])}", (row['anio'], row['n']),
                textcoords='offset points', xytext=(0, offset),
                ha='center', fontsize=11, fontweight='bold', color=color)

ax.axhline(media, color='gray', linestyle=':', linewidth=1.2,
           label=f'Promedio: {media:.0f} víctimas/año')
ax.axvspan(2019.7, 2020.9, alpha=0.1, color='#4dac26',
           label='ASPO · COVID-19 (mínimo: 81)')

ax.set_xlabel('Año'); ax.set_ylabel('Víctimas fatales')
ax.set_title('Evolución anual de víctimas fatales en siniestros viales · CABA 2019–2024',
             fontweight='bold')
ax.set_xticks(por_anio['anio']); ax.set_ylim(55, 130)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("Variación interanual:")
for anio, v in por_anio.set_index('anio')['n'].pct_change().dropna().items():
    print(f"  {'▼' if v<0 else '▲'} {anio}: {v:+.1%}")

**Lectura:** El mínimo histórico fue **2020 con 81 víctimas** — una caída del 22% respecto a 2019, explicada por el ASPO: con circulación mínima, los siniestros cayeron en proporción directa. Esto es un experimento natural que confirma que los siniestros fatales son función del volumen de tránsito. Preocupante: **2024 ya registra 113 víctimas**, el máximo del período post-pandemia, señal de que sin intervención estructural el sistema escala con el tránsito.

### 4.2 · Distribución por modo de desplazamiento

In [ ]:
conteo = df['modo'].value_counts()
pct    = (conteo / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4.5))
colores = [COLOR_PRIM if i < 4 else '#aec6d4' for i in range(len(conteo))]
bars = ax.barh(conteo.index[::-1], conteo.values[::-1],
               color=colores[::-1], edgecolor='white', height=0.65)

for bar, n, p in zip(bars, conteo.values[::-1], pct.values[::-1]):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
            f'{n}  ({p}%)', va='center', fontsize=10, color='#333')

ax.set_xlabel('Víctimas fatales (2019–2024)')
ax.set_title('¿En qué vehículo viajaban las víctimas fatales?', fontweight='bold', pad=12)
ax.set_xlim(0, conteo.max() * 1.3)
plt.tight_layout(); plt.show()

print(pd.DataFrame({'Víctimas': conteo, 'Porcentaje (%)': pct}).to_string())

**Lectura:** Los **motociclistas representan el 42% de las víctimas fatales** (256 de 610), casi igualados por los peatones con el 39% (238). Juntos concentran el **81% de todas las muertes viales**. Esta concentración tan marcada en solo dos modos no es casual: la moto carece de carrocería protectora y el peatón no tiene ninguna. El auto, a pesar de ser el vehículo más numeroso en la ciudad, representa solo el 8.7% de las víctimas fatales — evidencia directa del efecto protector de la carrocería.

### 4.3 · Distribución por sexo

In [ ]:
df_s = df[df['sexo'].isin(['M','F'])]
conteo_s = df_s['sexo'].map({'M':'MASCULINO','F':'FEMENINO'}).value_counts()
pct_s    = (conteo_s / len(df_s) * 100).round(1)

# % masculino por modo
pct_masc_modo = (
    df[df['sexo'].isin(['M','F'])]
    .groupby('modo')['sexo']
    .apply(lambda x: (x=='M').mean()*100)
    .sort_values(ascending=True)
)
# Solo modos con n >= 5
conteo_modo = df['modo'].value_counts()
modos_validos = conteo_modo[conteo_modo >= 5].index
pct_masc_modo = pct_masc_modo[pct_masc_modo.index.isin(modos_validos)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Global
bars = axes[0].bar(['MASCULINO','FEMENINO'], conteo_s.values,
                    color=[COLOR_PRIM, COLOR_SEC], edgecolor='white', width=0.5)
axes[0].bar_label(bars,
    labels=[f'{n:,}\n({p}%)' for n, p in zip(conteo_s.values, pct_s.values)],
    padding=5, fontsize=11)
axes[0].set_title('Distribución global por sexo', fontweight='bold')
axes[0].set_ylabel('Víctimas fatales')
axes[0].set_ylim(0, conteo_s.max() * 1.22)

# Por modo
avg = (df_s['sexo']=='M').mean()*100
colores_b = [COLOR_SEC if v >= avg else COLOR_PRIM for v in pct_masc_modo.values]
bars2 = axes[1].barh(pct_masc_modo.index, pct_masc_modo.values,
                      color=colores_b, edgecolor='white', height=0.6)
axes[1].axvline(avg, color='gray', linestyle='--', linewidth=1.5,
                label=f'Promedio: {avg:.1f}%')
axes[1].bar_label(bars2, fmt='%.0f%%', padding=3, fontsize=9)
axes[1].set_xlabel('% masculino'); axes[1].set_xlim(0, 110)
axes[1].set_title('% de víctimas masculinas por modo', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Distribución por sexo — víctimas fatales CABA 2019–2024',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print(f"Masculino: {pct_s['MASCULINO']:.1f}% del total")
print("\n% masculino por modo:")
print(pct_masc_modo.sort_values(ascending=False).round(1).to_string())

**Lectura:** El **74.8% de las víctimas son hombres** (3 de cada 4). La brecha es más pronunciada en motociclistas (88% masculinos), coherente con que los hombres dominan el uso de motos tanto para transporte personal como para delivery. Los **peatones son el grupo más equilibrado** (58% masculinos), reflejando que la vulnerabilidad peatonal no depende del sexo sino de la exposición al tránsito. Incluso en bicicleta (76% masculinos), la brecha persiste.

### 4.4 · Distribución de edad

In [ ]:
df_e = df.dropna(subset=['edad'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Histograma global
axes[0].hist(df_e['edad'], bins=20, color=COLOR_PRIM, edgecolor='white', alpha=0.85)
axes[0].axvline(df_e['edad'].mean(),   color=COLOR_SEC,   linestyle='--', lw=2,
                label=f"Media = {df_e['edad'].mean():.0f} años")
axes[0].axvline(df_e['edad'].median(), color='#4dac26', linestyle=':',  lw=2,
                label=f"Mediana = {df_e['edad'].median():.0f} años")
axes[0].set_xlabel('Edad'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de edad — todas las víctimas fatales', fontweight='bold')
axes[0].legend()

# Boxplot por modo
modos_top = ['MOTO','PEATON','AUTO','BICICLETA']
df_box = df[df['modo'].isin(modos_top)].dropna(subset=['edad'])
orden  = df_box.groupby('modo')['edad'].median().sort_values().index

bp = axes[1].boxplot(
    [df_box[df_box['modo']==m]['edad'].values for m in orden],
    labels=orden, patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
    flierprops=dict(marker='o', alpha=0.3, markersize=3)
)
for patch, color in zip(bp['boxes'], PALETA[:len(orden)]):
    patch.set_facecolor(color)
axes[1].set_xlabel('Modo de desplazamiento'); axes[1].set_ylabel('Edad')
axes[1].set_title('Distribución de edad por modo de víctima', fontweight='bold')

plt.suptitle('Perfil etario de las víctimas fatales · CABA 2019–2024',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print("Estadísticos por modo:")
print(df_box.groupby('modo')['edad']
      .agg(['mean','median','min','max','count'])
      .round(1).sort_values('median').to_string())

**Lectura:** La edad mediana global es **39 años** — adultos en plena edad productiva. Pero el boxplot revela que esta mediana global encubre dos poblaciones completamente distintas:

- **Motociclistas:** mediana 31 años (IQR 22–43). Jóvenes, muchos en contexto laboral de delivery.
- **Peatones:** mediana 57 años (IQR 38–72). Adultos mayores, con mayor tiempo de cruce necesario y menor capacidad de reacción.
- **Bicicleta:** mediana 47 años — perfil de adulto en uso utilitario o recreativo.
- **Auto:** mediana 40 años, el más heterogéneo de los cuatro grupos.

La brecha de **26 años** entre la mediana de motociclistas y peatones define dos crisis de seguridad vial distintas que coexisten en la misma ciudad.

## 5. Análisis en Profundidad

### 5.1 · Evolución por modo — ¿la composición cambia con el tiempo?

In [ ]:
modos_top = ['MOTO','PEATON','AUTO','BICICLETA']
df_ev     = df[df['modo'].isin(modos_top)]
por_at    = df_ev.groupby(['anio','modo']).size().unstack(fill_value=0)[modos_top]
por_at_p  = por_at.div(por_at.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for modo, color in zip(modos_top, PALETA[:4]):
    axes[0].plot(por_at.index, por_at[modo], marker='o', label=modo,
                 color=color, linewidth=2.2)
    axes[1].plot(por_at_p.index, por_at_p[modo], marker='o', label=modo,
                 color=color, linewidth=2.2)

for ax in axes:
    ax.axvspan(2019.7, 2020.9, alpha=0.08, color='green')
    ax.set_xticks(por_at.index); ax.legend(fontsize=9); ax.set_xlabel('Año')

axes[0].set_ylabel('Víctimas fatales'); axes[0].set_title('Absolutos', fontweight='bold')
axes[1].set_ylabel('Proporción'); axes[1].set_title('Composición relativa', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.suptitle('Evolución de la composición de víctimas fatales · CABA 2019–2024',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

print("Proporción 2019 vs. 2024:")
for m in modos_top:
    p19 = por_at_p.loc[2019, m]
    p24 = por_at_p.loc[2024, m]
    print(f"  {m:<12}: {p19:.1%} → {p24:.1%}  ({p24-p19:+.1%})")

**Lectura:** La composición de víctimas **no es estable**. El hallazgo más relevante: en 2024 los **peatones casi alcanzan a los motociclistas** en términos de participación relativa. Esto puede responder al crecimiento del tránsito vehicular post-pandemia y a la mayor exposición peatonal en zonas de alta densidad. Los **ciclistas** muestran una tendencia creciente desde 2022, consistente con la expansión de la infraestructura ciclista pero también con el aumento del uso de bicicleta.

### 5.2 · Perfil comparado: Motociclistas vs. Peatones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, modo, color, desc in zip(
    axes, ['MOTO','PEATON'], [COLOR_PRIM, COLOR_SEC],
    ['jóvenes conductores y riders de delivery',
     'adultos mayores en cruces peatonales']
):
    sub    = df[df['modo'] == modo]['edad'].dropna()
    media  = sub.mean()
    mediana = sub.median()

    ax.hist(sub, bins=15, color=color, edgecolor='white', alpha=0.82)
    ax.axvline(media,   color='black',   linestyle='--', lw=2,
               label=f'Media = {media:.0f} años')
    ax.axvline(mediana, color='#555', linestyle=':', lw=2,
               label=f'Mediana = {mediana:.0f} años')
    ax.set_xlabel('Edad'); ax.set_ylabel('Frecuencia')
    ax.set_title(f'{modo} — {desc}', fontweight='bold')
    ax.legend(fontsize=9)

    # Anotar tramos de edad
    n_total  = len(sub)
    n_jovenes = (sub < 35).sum()
    n_mayores = (sub >= 60).sum()
    ax.text(0.97, 0.92, f'< 35 años: {n_jovenes/n_total:.0%}',
            transform=ax.transAxes, ha='right', fontsize=9, color='#333')
    ax.text(0.97, 0.84, f'≥ 60 años: {n_mayores/n_total:.0%}',
            transform=ax.transAxes, ha='right', fontsize=9, color='#333')

plt.suptitle('Los dos grupos más vulnerables — perfil etario comparado',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

**Lectura:** Los histogramas ilustran el contraste más importante del dataset:

- **Motociclistas:** el 54% tiene menos de 35 años. La distribución está fuertemente sesgada hacia la izquierda — son mayoritariamente adultos jóvenes. En muchos casos, trabajadores del delivery cuya herramienta de trabajo es la moto.

- **Peatones:** la distribución es bimodal, con un pico en adultos activos y otro marcado en adultos mayores (60+ años). El 38% tiene 60 años o más. Un adulto mayor necesita más tiempo para cruzar y tiene menor capacidad de reacción ante vehículos — y muchas intersecciones no están diseñadas para eso.

### 5.3 · Cruces de variables: modo × sexo × edad

In [ ]:
# Tabla resumen por modo
resumen = []
for modo in ['MOTO','PEATON','AUTO','BICICLETA']:
    sub    = df[df['modo'] == modo]
    sub_s  = sub[sub['sexo'].isin(['M','F'])]
    sub_e  = sub['edad'].dropna()
    resumen.append({
        'Modo':            modo,
        'N':               len(sub),
        '% del total':     f"{len(sub)/len(df)*100:.1f}%",
        '% masculino':     f"{(sub_s['sexo']=='M').mean()*100:.0f}%",
        'Edad media':      f"{sub_e.mean():.0f}",
        'Edad mediana':    f"{sub_e.median():.0f}",
        '% < 35 años':     f"{(sub_e < 35).mean()*100:.0f}%",
        '% ≥ 60 años':     f"{(sub_e >= 60).mean()*100:.0f}%",
    })

df_resumen = pd.DataFrame(resumen).set_index('Modo')
print("=== Tabla resumen por modo ===")
print(df_resumen.to_string())

**Lectura:** La tabla integra los hallazgos en una vista comparada. Los datos confirman la bipolaridad del problema:

- **Motos:** grupo más grande (42%), el más joven (mediana 31), el más masculinizado (88%), con el 54% menor de 35 años.
- **Peatones:** segundo grupo (39%), el más envejecido (mediana 57), el más equilibrado en sexo (58% M), con el 38% de 60+ años.
- **Bicicleta:** en crecimiento, perfil de adulto medio (mediana 47), fuertemente masculinizado (76%).
- **Auto:** el grupo más heterogéneo en todos los indicadores.

### 5.4 · Estacionalidad mensual

In [ ]:
por_mes = df.groupby('mes').size().reset_index(name='n')
meses_labels = ['Ene','Feb','Mar','Abr','May','Jun',
                 'Jul','Ago','Sep','Oct','Nov','Dic']

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(por_mes['mes'], por_mes['n'],
               color=[COLOR_SEC if n == por_mes['n'].max() else COLOR_PRIM
                      for n in por_mes['n']],
               edgecolor='white', alpha=0.88)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_labels)
ax.axhline(por_mes['n'].mean(), color='gray', linestyle=':', linewidth=1.5,
           label=f"Promedio mensual: {por_mes['n'].mean():.0f}")
ax.set_xlabel('Mes'); ax.set_ylabel('Víctimas fatales')
ax.set_title('Distribución mensual de víctimas fatales · CABA 2019–2024 (acumulado)',
             fontweight='bold')
ax.legend(); ax.set_ylim(0, por_mes['n'].max() * 1.15)
plt.tight_layout(); plt.show()

print("Mes con más víctimas:", meses_labels[por_mes.loc[por_mes['n'].idxmax(), 'mes']-1],
      "(", por_mes['n'].max(), ")")
print("Mes con menos víctimas:", meses_labels[por_mes.loc[por_mes['n'].idxmin(), 'mes']-1],
      "(", por_mes['n'].min(), ")")

**Lectura:** La distribución mensual no muestra una estacionalidad marcada — las muertes viales ocurren todo el año. Sin embargo, los meses de **mayor tránsito y actividad** (octubre–diciembre, marzo–mayo) tienden a concentrar algo más de víctimas. El verano porteño (enero–febrero) muestra una leve baja, coherente con la reducción de tránsito por vacaciones. Este patrón refuerza la relación directa entre volumen vehicular y siniestralidad fatal.

## 6. Insights Clave

---

### 🔴 Insight 1 — Motos y peatones concentran el 81% de las muertes viales

256 motociclistas (42%) y 238 peatones (39%) representan 4 de cada 5 víctimas fatales de CABA entre 2019 y 2024. Que dos modos concentren tal proporción no es casualidad: ambos carecen de protección física ante impactos. **El sistema vial está diseñado para la seguridad del vehículo, no de la persona.**

---

### 🟠 Insight 2 — 2024 es el máximo post-pandemia y la tendencia es ascendente

Con 113 víctimas en 2024, la siniestralidad fatal supera el promedio del período (101.7) y muestra una tendencia creciente desde 2020. El ASPO demostró que la reducción masiva es posible — la curva actual muestra que sin intervención la siniestralidad escala con el tránsito.

---

### 🟡 Insight 3 — Los motociclistas mueren jóvenes: mediana de 31 años, 88% hombres

El perfil del motociclista fatal es un hombre joven (el 54% tiene menos de 35 años). Este perfil coincide exactamente con el del trabajador de delivery — un sector que creció exponencialmente desde 2019 con Rappi, PedidosYa y otras apps. **Morir en moto camino a entregar un pedido es un riesgo laboral encubierto.**

---

### 🟢 Insight 4 — Los peatones fallecen a edades mucho mayores: mediana de 57 años

El 38% de los peatones fallecidos tiene 60 años o más. Un adulto mayor necesita entre 6 y 8 segundos para cruzar una avenida de 4 carriles — muchos semáforos de CABA no lo permiten. Esta no es una crisis de velocidad sino de **diseño urbano que no contempla el envejecimiento de la población**.

---

### 🔵 Insight 5 — El sesgo de género es transversal pero tiene matices importantes

El 74.8% de las víctimas son hombres, pero la brecha varía por modo: desde el 88% en motos hasta el 58% en peatones. La menor brecha en peatones confirma que el riesgo peatonal no depende del comportamiento individual (que difiere por género) sino de la infraestructura — las avenidas son peligrosas para todos.

---

### ⚪ Insight 6 — La bicicleta crece como modo de víctimas fatales

Con 42 víctimas en el período y una tendencia creciente desde 2022, los ciclistas emergen como un grupo de riesgo creciente. La expansión de ciclovías no ha sido suficiente para contener la siniestralidad — en muchos tramos, la bici comparte espacio con vehículos pesados sin protección adecuada.

## 7. Conclusiones y Recomendaciones

### ¿Quiénes son los más vulnerables?

Dos grupos, con perfiles radicalmente distintos, concentran el 81% de las muertes:

**Motociclistas jóvenes (18–35 años, 88% hombres)** — trabajadores y usuarios de un vehículo sin protección. La moto se convirtió en herramienta de trabajo masiva sin que el marco regulatorio ni la infraestructura se adaptaran.

**Peatones adultos mayores (57 años de mediana)** — ciudadanos cruzando calles que no están diseñadas para sus tiempos de reacción y velocidades de desplazamiento.

### Recomendaciones de política pública

| Medida | Grupo objetivo | Base en los datos |
|---|---|---|
| Regulación laboral del delivery: seguro obligatorio, horas máximas, capacitación | Motociclistas 18–35 | 42% víctimas, 88% masculinos, mediana 31 años |
| Ciclos semafóricos más largos en avenidas de alta densidad peatonal | Peatones adultos mayores | 38% de peatones tiene 60+ años |
| Infraestructura ciclista segregada en corredores de alta siniestralidad | Ciclistas | Tendencia creciente 2022–2024 |
| Control de velocidad en zonas de alto flujo peatonal | Peatones en general | 39% del total de víctimas |
| Monitoreo anual de la proporción moto/total como indicador de gestión | Todos | Crecimiento sostenido |

### Próximos pasos de análisis

- **Incorporar datos de hechos (no solo víctimas):** el dataset complementario contiene ubicación geográfica — cruzarlos permitiría identificar intersecciones puntuales de alta siniestralidad.
- **Análisis de gravedad relativa:** comparar los perfiles de víctimas leves, graves y mortales para identificar qué factores determinan la gravedad del desenlace.
- **Modelo predictivo de gravedad:** con las variables disponibles (modo, sexo, edad, rol), entrenar un clasificador binario LEVE/MORTAL y analizar qué variables tienen mayor poder predictivo.
- **Cruce con datos de tránsito:** normalizar por volumen vehicular para obtener una tasa de riesgo real por modo.

---
*Datos: Observatorio de Movilidad y Seguridad Vial (OMSV) · GCBA · Buenos Aires Data*  
*Dataset: Siniestros viales — victimas · 2019–2024 · Licencia CC Attribution*